# Condition B — Fine-tune (answers only)

Fine-tune Flan-T5-base on WTQ; target = final answer. 2 seeds. Saves checkpoints for the generalization test. Expected to give the highest WTQ accuracy.

In [ ]:
# --- Setup: on Colab this clones the repo and installs deps; locally it is a no-op ---
import os
if not os.path.isdir("src"):
    if not os.path.isdir("ECS111FinalProject"):
        !git clone https://github.com/adiseshvsanklapur/ECS111FinalProject.git
    os.chdir("ECS111FinalProject")
    !pip -q install -r requirements.txt
print("cwd:", os.getcwd())

In [ ]:
# Set SMOKE = False for the full, reported run. SMOKE = True does a fast real
# end-to-end pass (flan-t5-small, tiny slice) to confirm everything works first.
SMOKE = True

In [ ]:
from src import config
device = config.get_device()
print("device:", device)

if SMOKE:
    prompt_models = [config.SMOKE_MODEL]
    seeds = [13]
    eval_n = config.SMOKE_EVAL_N
    eval_n_cot = config.SMOKE_EVAL_N
    train_n = config.SMOKE_TRAIN_N
else:
    prompt_models = config.PROMPT_MODELS      # flan-t5-base + large
    seeds = config.SEEDS                       # [13, 42]
    eval_n = config.EVAL_N                      # 1000
    eval_n_cot = config.EVAL_N_COT             # 500
    train_n = config.TRAIN_N                    # 8000

In [ ]:
import os
from src.data import load_wtq_train, load_wtq_eval
from src.prompts import build_train_source, build_train_target, build_baseline_prompt
from src.trainer import train
from src.evaluate import predict_and_evaluate

os.makedirs("checkpoints", exist_ok=True)
ft_label = config.SMOKE_MODEL if SMOKE else config.FINETUNE_MODEL

rows = []
for seed in seeds:
    train_ex = load_wtq_train(n=train_n, seed=seed)
    sources = [build_train_source(e) for e in train_ex]
    targets = [build_train_target(e) for e in train_ex]
    model, tok, device = train(
        config.FINETUNE_MODEL, sources, targets, seed=seed, smoke=SMOKE, device=device
    )
    ckpt = f"checkpoints/finetune_answers_seed{seed}"
    model.save_pretrained(ckpt); tok.save_pretrained(ckpt)

    eval_ex = load_wtq_eval(n=eval_n, seed=config.EVAL_SEED)
    res = predict_and_evaluate(
        model, tok, eval_ex, build_baseline_prompt,
        condition="finetune_answers", model_id=ft_label, seed=seed,
        task="wtq", device=device,
    )
    print("seed", seed, res["metrics"])
    rows.append((seed, res["metrics"]))
rows